# Learning from data: regression, gradient descent, k-means

**Course**: Artificial Intelligence · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Chapter 3**: Types of AI and approaches · Some machine learning algorithms
**Duration**: about 1 h 30

### What you will do
- Redo in Python the calculations of exercise 2 a of the chapter 3 tutorial (TD): the regression line by the least squares method, then check it with NumPy and scikit-learn.
- Follow gradient descent step by step (exercise 2 b), see the role of the learning rate, then use it to make the machine "learn" the regression line.
- Work through k-means by hand (exercise 2 d), then compare with scikit-learn's `KMeans` and the elbow method.

### How to use this notebook
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. **You never have to write code.** Change only the fields of the cells titled "TO CHANGE".

- In Google Colab, the menu "Runtime → Run all" runs the whole notebook; the "TO CHANGE" fields appear as sliders or text boxes.
- In Jupyter, these fields are plain `#@param` comments: change only the value after the `=` sign.
- After changing a value, run the cell again, then the cells that follow it.

Keep your tutorial solution and a calculator at hand: before each cell, **predict** the result; afterwards, compare with the "What you should see" section. All the data in this notebook are **made up** and tiny, so that the calculations can be redone by hand.

## 1. Linear regression (TD 3, exercise 2 a)
Linear regression is **supervised learning**: for each example, we know a feature `x` and the right answer `y`, and we look for the line ŷ = a·x + b that comes as close as possible to the points. For five students (made-up data), we recorded the number of hours `x` spent reading each week and the number `y` of books read during the semester: (1, 2), (2, 3), (3, 5), (4, 4), (5, 6).

The least squares method gives:

a = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)²  and  b = ȳ − a·x̄

The cell computes the means x̄ and ȳ, then the table of deviations from the solution, column by column. With `pandas`, a calculation written on a column applies to all its rows. The `books_student_5` slider lets you change the value for the fifth student (6 in the tutorial).

**Before running:** what are x̄ and ȳ? What is the sum of the column (x − x̄)(y − ȳ)?

In [ ]:
#@title TO CHANGE: the data of exercise 2 a and the table of deviations
books_student_5 = 6  #@param {type:"slider", min:0, max:15, step:1}

import pandas as pd
x = [1, 2, 3, 4, 5]                        # reading hours per week (made up)
y = [2, 3, 5, 4, books_student_5]          # books read during the semester (made up)
x_mean = sum(x) / len(x)
y_mean = sum(y) / len(y)
print("x̄ =", x_mean, "  ȳ =", y_mean)
table = pd.DataFrame({"x": x, "y": y})
table["x - x̄"] = table["x"] - x_mean                      # deviation from the mean, row by row
table["y - ȳ"] = table["y"] - y_mean
table["(x - x̄)(y - ȳ)"] = table["x - x̄"] * table["y - ȳ"]
table["(x - x̄)²"] = table["x - x̄"] ** 2
table

### What you should see
`x̄ = 3.0   ȳ = 4.0`, then the table from the solution:

| | x | y | x - x̄ | y - ȳ | (x - x̄)(y - ȳ) | (x - x̄)² |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | -2.0 | -2.0 | 4.0 | 4.0 |
| 1 | 2 | 3 | -1.0 | -1.0 | 1.0 | 1.0 |
| 2 | 3 | 5 | 0.0 | 1.0 | 0.0 | 0.0 |
| 3 | 4 | 4 | 1.0 | 0.0 | 0.0 | 1.0 |
| 4 | 5 | 6 | 2.0 | 2.0 | 4.0 | 4.0 |

**Observe:** add up each column in your head. The columns x − x̄ and y − ȳ always add up to zero: why?

### Calculating a and b
`sum()` adds up a column. We then apply the two formulas from the course.

**Before running:** according to your solution, what are a and b?

In [ ]:
#@title Calculate a and b by the least squares method
sum_products = table["(x - x̄)(y - ȳ)"].sum()
sum_squares = table["(x - x̄)²"].sum()
a = sum_products / sum_squares
b = y_mean - a * x_mean
print("Σ(x - x̄)(y - ȳ) =", sum_products, "   Σ(x - x̄)² =", sum_squares)
print(f"a = {a:.2f}   b = {b:.2f}")
print(f"Regression line: ŷ = {a:.2f} x + {b:.2f}")

### What you should see
```
Σ(x - x̄)(y - ȳ) = 9.0    Σ(x - x̄)² = 10.0
a = 0.90   b = 1.30
Regression line: ŷ = 0.90 x + 1.30
```
This is the tutorial solution: a = 9 / 10 = 0.9 and b = 4 − 0.9 × 3 = 1.3. In these made-up data, each extra hour of reading per week is associated with 0.9 more books over the semester.

**Observe:** "associated with" does not mean "causes". Does reading for more hours make students read more books, or do students who love reading simply do both?

### Predict, then check the differences
The model is used to **predict**: for `x` hours, ŷ = a·x + b. The cell also computes the predicted values ŷ for the five students and the differences y − ŷ (the **residuals**), as in the check in the solution.

**Before running:** how many books does the model predict for 6 hours a week? What is the sum of the differences?

In [ ]:
#@title TO CHANGE: predict for a number of hours, then check the differences
hours = 6  #@param {type:"slider", min:0, max:12, step:1}

print(f"Prediction for {hours} h a week: ŷ = {a * hours + b:.1f} books")
table["ŷ"] = a * table["x"] + b                     # the predicted values
table["y - ŷ"] = table["y"] - table["ŷ"]            # the differences (residuals)
print("Sum of the differences:", round(table["y - ŷ"].sum(), 2))
print("Sum of the squared differences:", round((table["y - ŷ"] ** 2).sum(), 2))
table[["x", "y", "ŷ", "y - ŷ"]].round(2)

### What you should see
```
Prediction for 6 h a week: ŷ = 6.7 books
Sum of the differences: 0.0
Sum of the squared differences: 1.9
```
| | x | y | ŷ | y - ŷ |
|---|---|---|---|---|
| 0 | 1 | 2 | 2.2 | -0.2 |
| 1 | 2 | 3 | 3.1 | -0.1 |
| 2 | 3 | 5 | 4.0 | 1.0 |
| 3 | 4 | 4 | 4.9 | -0.9 |
| 4 | 5 | 6 | 5.8 | 0.2 |

The predicted values are 2.2, 3.1, 4.0, 4.9 and 5.8. The sum of the differences is zero, as always with least squares, and the sum of their squares is 1.9: no other line does better.

**Modify:** move the `hours` slider to 12, then to 0. The model predicts 12.1 books, then 1.3 books for a student who never reads. Why are these **extrapolations**, outside the observed range (1 to 5 hours), doubtful?

### Check with NumPy and scikit-learn
The libraries do the same calculation in one line. `numpy.polyfit(x, y, 1)` finds the least squares polynomial of degree 1 (a straight line). With scikit-learn, you create a `LinearRegression` model, train it with `fit`, then predict with `predict`: this is the vocabulary of all machine learning.

**Before running:** will the three methods give the same a and b?

In [ ]:
#@title Check with NumPy and scikit-learn
import numpy as np
from sklearn.linear_model import LinearRegression

slope, intercept = np.polyfit(x, y, 1)          # least squares line (degree 1)
print(f"NumPy        : a = {slope:.2f}   b = {intercept:.2f}")
model = LinearRegression()
model.fit([[value] for value in x], y)          # train: scikit-learn expects a list of rows
print(f"scikit-learn : a = {model.coef_[0]:.2f}   b = {model.intercept_:.2f}")
print(f"scikit-learn prediction for 6 h: {model.predict([[6]])[0]:.1f} books")

### What you should see
```
NumPy        : a = 0.90   b = 1.30
scikit-learn : a = 0.90   b = 1.30
scikit-learn prediction for 6 h: 6.7 books
```
The three methods agree. The calculation "by hand" in the earlier cells is not a pointless step: it is exactly what the libraries do.

### The scatter plot and the line
`plt.scatter` places the points, `plt.plot` draws the line; the orange star marks the prediction for the number of hours chosen above.

**Before running:** which student will be farthest from the line?

In [ ]:
#@title Draw the scatter plot and the regression line
import matplotlib.pyplot as plt

x_values = np.linspace(0, max(6, hours) + 1, 50)       # x values used to draw the line
plt.figure(figsize=(6, 4))
plt.plot(x_values, a * x_values + b, color="#eb6834", linewidth=2, label=f"ŷ = {a:.2f} x + {b:.2f}")
plt.scatter(x, y, color="#2a78d6", s=60, zorder=3, label="Students (made-up data)")
plt.scatter([hours], [a * hours + b], color="#eb6834", marker="*", s=250, zorder=3, label=f"Prediction for {hours} h")
plt.xlabel("Reading hours per week (x)")
plt.ylabel("Books read during the semester (y)")
plt.title("Linear regression (TD 3, exercise 2 a)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### What you should see
Five blue points, a rising orange line and a star at (6, 6.7). The third student (3, 5) is the farthest from the line: their difference is 1.0.

**Modify:** at the top of part 1, move `books_student_5` to 12, then run the cells of part 1 again in order. What happens to a and b? Is one unusual student enough to change the line? What should you conclude about a survey of just a few people?

## 2. Gradient descent (TD 3, exercise 2 b)
Most models cannot be calculated in one go: their parameters are found **step by step**. Imagine a hiker in the fog: they cannot see the valley, but they can feel the slope under their feet, and each time they take a step downhill. The valley is the **error function** J(w); the slope is its derivative J′(w). At each step:

w ← w − η · J′(w)

η (eta) is the **learning rate**, the length of the step. Take J(w) = (w − 3)², which is smallest at w = 3, with derivative J′(w) = 2(w − 3), w₀ = 0 and η = 0.1.

**Before running:** calculate w₁, w₂, J(w₀), J(w₁) and J(w₂). Toward which value does the sequence move?

In [ ]:
#@title TO CHANGE: gradient descent, step by step
eta = 0.1  #@param {type:"slider", min:0.01, max:1.2, step:0.01}
w_start = 0.0  #@param {type:"number"}
number_of_steps = 5  #@param {type:"slider", min:1, max:30, step:1}

w = w_start
path = [w]                                    # keep each value of w for the chart
print(f"step 0: w = {w:.4f}   J(w) = {(w - 3) ** 2:.4f}")
for step in range(1, number_of_steps + 1):
    slope = 2 * (w - 3)                       # the derivative J'(w) at point w
    w = w - eta * slope                       # one step downhill
    path.append(w)
    print(f"step {step}: w = {w:.4f}   J(w) = {(w - 3) ** 2:.4f}   (slope used: {slope:.3f})")

### What you should see
```
step 0: w = 0.0000   J(w) = 9.0000
step 1: w = 0.6000   J(w) = 5.7600   (slope used: -6.000)
step 2: w = 1.0800   J(w) = 3.6864   (slope used: -4.800)
step 3: w = 1.4640   J(w) = 2.3593   (slope used: -3.840)
step 4: w = 1.7712   J(w) = 1.5099   (slope used: -3.072)
step 5: w = 2.0170   J(w) = 0.9664   (slope used: -2.458)
```
This is the tutorial solution: w₁ = 0.6 and w₂ = 1.08; J(w₀) = 9, J(w₁) = 5.76 and J(w₂) = 3.6864. The error decreases at each step, and w moves toward 3. At each step, the distance to the minimum is multiplied by 1 − 2η = 0.8: 3, 2.4, 1.92, 1.536…

**Modify:** move `number_of_steps` to 30: does w reach exactly 3? Then try `eta` = 1 (w oscillates endlessly between 0 and 6), `eta` = 1.1 (w jumps over the minimum, 6.6 then −1.32 then 8.18, and moves further away: the algorithm **diverges**) and `eta` = 0.01 (w only reaches 0.29 after five steps: learning is very slow).

### See the steps on the error curve
The blue curve is the error function J(w); the orange points are the successive values of w calculated by the previous cell.

**Before running:** will the steps get bigger or smaller? Why?

In [ ]:
#@title Draw the curve J(w) and the steps of the descent
w_values = np.linspace(-2, 8, 200)
plt.figure(figsize=(6, 4))
plt.plot(w_values, (w_values - 3) ** 2, color="#2a78d6", linewidth=2, label="J(w) = (w - 3)²")
errors = [(v - 3) ** 2 for v in path]
plt.plot(path, errors, color="#eb6834", marker="o", linewidth=1, label=f"Steps of the descent (η = {eta})")
plt.xlabel("Value of the parameter w")
plt.ylabel("Error J(w)")
plt.title("Gradient descent (TD 3, exercise 2 b)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### What you should see
A blue parabola with its lowest point at w = 3, and six orange points going down the left side, from w = 0 to w ≈ 2. The steps get shorter, because the slope becomes gentler near the bottom: the hiker slows down on their own.

**Modify:** with `eta` = 1.1 in the previous cell, run both cells again: the points jump from one side to the other while climbing.

### Make the machine "learn" the regression line
Let us connect parts 1 and 2. The error of the line ŷ = a·x + b is the mean of the squared differences, J(a, b) = (1/n) · Σ(y − ŷ)². Let us start from a flat line (a = 0, b = 0) and apply the same rule, w ← w − η · slope, to each of the two parameters. The formulas for the two slopes are written in the cell: `np.mean` takes the mean over the five students. We display steps 1, 10, 100 and the last one.

**Before running:** after 1,000 steps, which values of a and b will the machine have found?

In [ ]:
#@title TO CHANGE: find a and b again by gradient descent
eta_line = 0.05  #@param {type:"number"}
number_of_steps_line = 1000  #@param {type:"slider", min:10, max:2000, step:10}

X = np.array(x, dtype=float)
Y = np.array(y, dtype=float)
a_learned = 0.0                                  # start from a flat line
b_learned = 0.0
for step in range(1, number_of_steps_line + 1):
    differences = Y - (a_learned * X + b_learned)   # the errors of the current line
    a_learned = a_learned - eta_line * (-2 * np.mean(differences * X))   # slope of J with respect to a
    b_learned = b_learned - eta_line * (-2 * np.mean(differences))       # slope of J with respect to b
    if step in (1, 10, 100) or step == number_of_steps_line:
        error = np.mean((Y - (a_learned * X + b_learned)) ** 2)
        print(f"step {step:4d}: a = {a_learned:.4f}   b = {b_learned:.4f}   error J = {error:.4f}")

### What you should see
```
step    1: a = 1.3800   b = 0.4000   error J = 1.1324
step   10: a = 1.1279   b = 0.4771   error J = 0.5033
step  100: a = 0.9491   b = 1.1226   error J = 0.3857
step 1000: a = 0.9000   b = 1.3000   error J = 0.3800
```
Step by step, the machine finds a = 0.9 and b = 1.3 again, without knowing the least squares formulas. The final error, 0.38, is the sum of the squared differences (1.9) divided by the 5 students. For a model, **learning** means exactly this: adjusting its parameters to reduce its error on examples. A large language model does the same, with billions of parameters instead of two.

**Modify:** try `number_of_steps_line` = 100 (the line has not quite been found yet), then `eta_line` = 0.1: a, b and the error become gigantic (numbers with more than a hundred digits), because the step is too large. The algorithm diverges.

## 3. k-means (TD 3, exercise 2 d)
Clustering is **unsupervised learning**: there are no labels; we look for groups of similar objects. The k-means algorithm alternates two operations: **assign** each point to the nearest center, then **recompute** each center as the mean of the points in its group.

Six characters in a play have 1, 3, 4, 9, 10 and 14 lines respectively (made-up data). We look for k = 2 groups, starting from the centers μ₁ = 1 and μ₂ = 4. In one dimension, the distance between two numbers is the absolute value of their difference, `abs()`.

**Before running:** fill in the table for iteration 1 on paper. Which group does point 3 go to?

In [ ]:
#@title TO CHANGE: the starting centers, then iteration 1 by hand
mu1_start = 1  #@param {type:"number"}
mu2_start = 4  #@param {type:"number"}

speeches = [1, 3, 4, 9, 10, 14]             # lines spoken by six characters (made up)
rows = []
for point in speeches:
    d1 = abs(point - mu1_start)             # distance to center 1
    d2 = abs(point - mu2_start)             # distance to center 2
    if d1 <= d2:
        group = 1                           # closer to μ1 (or a tie)
    else:
        group = 2
    rows.append([point, d1, d2, group])
pd.DataFrame(rows, columns=["Point", "Distance to μ1", "Distance to μ2", "Group"])

### What you should see
| | Point | Distance to μ1 | Distance to μ2 | Group |
|---|---|---|---|---|
| 0 | 1 | 0 | 3 | 1 |
| 1 | 3 | 2 | 1 | 2 |
| 2 | 4 | 3 | 0 | 2 |
| 3 | 9 | 8 | 5 | 2 |
| 4 | 10 | 9 | 6 | 2 |
| 5 | 14 | 13 | 10 | 2 |

This is the table from the solution. Update: group 1 = {1}, so μ₁ = 1; group 2 = {3, 4, 9, 10, 14}, so μ₂ = 40 / 5 = 8.

**Observe:** after this first iteration, group 2 still mixes very different roles (3 and 14 lines). Is one iteration enough?

### Repeat until the groups no longer change
The cell repeats the two operations. `[p for p in speeches if ...]` builds the list of the points that meet the condition. We stop when the centers no longer move: the groups are then stable.

**Before running:** how many iterations will be needed? What will the two final groups and their centers be?

In [ ]:
#@title Repeat assignment and center update
mu1 = mu1_start
mu2 = mu2_start
for iteration in range(1, 11):
    # 1. assign each point to the nearest center
    group1 = [p for p in speeches if abs(p - mu1) <= abs(p - mu2)]
    group2 = [p for p in speeches if abs(p - mu1) > abs(p - mu2)]
    old_centers = (mu1, mu2)
    # 2. each center becomes the mean of its group (if the group is not empty)
    if len(group1) > 0:
        mu1 = sum(group1) / len(group1)
    if len(group2) > 0:
        mu2 = sum(group2) / len(group2)
    print(f"Iteration {iteration}: {group1} and {group2} → μ1 = {mu1:.2f}, μ2 = {mu2:.2f}")
    if (mu1, mu2) == old_centers:
        print("The centers no longer move: the groups are stable, stop.")
        break

### What you should see
```
Iteration 1: [1] and [3, 4, 9, 10, 14] → μ1 = 1.00, μ2 = 8.00
Iteration 2: [1, 3, 4] and [9, 10, 14] → μ1 = 2.67, μ2 = 11.00
Iteration 3: [1, 3, 4] and [9, 10, 14] → μ1 = 2.67, μ2 = 11.00
The centers no longer move: the groups are stable, stop.
```
This is the solution, bonus included: in iteration 2, 3 and 4 join group 1; μ₁ = 8/3 ≈ 2.67 and μ₂ = 33/3 = 11. In iteration 3, nothing changes any more. Result: {1, 3, 4}, the minor roles, and {9, 10, 14}, the main roles. It is you, not the algorithm, who names the groups in this way.

**Modify:** start from `mu1_start` = 13 and `mu2_start` = 14 (run both cells again): do you get the same groups? In how many iterations?

### Check with scikit-learn's KMeans
`KMeans` does the same job. You can give it the starting centers of the tutorial (`init`), or let it draw random starting points several times (`n_init=10`) and keep the best solution. The **inertia** is the sum of the squared distances between each point and the center of its group: the smaller it is, the tighter the groups.

**Before running:** will scikit-learn find the same groups? Calculate the inertia: (1 − 8/3)² + (3 − 8/3)² + (4 − 8/3)² + (9 − 11)² + (10 − 11)² + (14 − 11)².

In [ ]:
#@title Check with scikit-learn's KMeans
from sklearn.cluster import KMeans

points = [[s] for s in speeches]                # scikit-learn expects a list of rows
start = [[mu1_start], [mu2_start]]              # the starting centers of the tutorial
km = KMeans(n_clusters=2, init=start, n_init=1).fit(points)
print("Tutorial start  → groups:", km.labels_, "| centers:", km.cluster_centers_.ravel().round(2))
print("Inertia:", round(km.inertia_, 2))
km_random = KMeans(n_clusters=2, n_init=10, random_state=0).fit(points)   # 10 random starts
print("Random starts   → groups:", km_random.labels_, "| centers:", km_random.cluster_centers_.ravel().round(2))

### What you should see
```
Tutorial start  → groups: [0 0 0 1 1 1] | centers: [ 2.67 11.  ]
Inertia: 18.67
Random starts   → groups: [1 1 1 0 0 0] | centers: [11.    2.67]
```
The same groups and the same centers, 2.67 and 11; the inertia is 18.67. With random starts, the group **numbers** are swapped (0 and 1 exchanged): these numbers are arbitrary, only the make-up of the groups matters. Depending on your version of scikit-learn, you may get either numbering.

### How many groups? The elbow method
The inertia always decreases as k increases (with 6 groups for 6 points, it would be 0). The **elbow method** keeps the value of k after which it hardly decreases any more.

**Before running:** where will the elbow be for the six characters?

In [ ]:
#@title The elbow method: inertia for each number of groups k
inertias = []
for k in range(1, 5):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(points)
    inertias.append(km_k.inertia_)
    print(f"k = {k}: inertia = {km_k.inertia_:.2f}")
plt.figure(figsize=(5, 3))
plt.plot(range(1, 5), inertias, color="#2a78d6", marker="o", linewidth=2)
plt.xticks(range(1, 5))
plt.xlabel("Number of groups k")
plt.ylabel("Inertia")
plt.title("Elbow method (lines of six characters)")
plt.grid(alpha=0.3)
plt.show()

### What you should see
```
k = 1: inertia = 122.83
k = 2: inertia = 18.67
k = 3: inertia = 5.17
k = 4: inertia = 1.00
```
and a curve that drops sharply from k = 1 to k = 2, then flattens: the elbow is at k = 2, which confirms the choice made in the tutorial.

**Observe:** with k = 3, does the main character (14 lines) get a group of their own? Is the drop in inertia worth a third group?

## Your turn
1. Part 1: move `books_student_5` to 12 and run part 1 again. Note a and b: does one unusual student change the line a lot?
2. Part 2: compare `eta` = 0.1, 0.5, 1 and 1.1 with 10 steps. Which value reaches the minimum fastest? Which one never reaches it?
3. Part 3: start from μ₁ = 13 and μ₂ = 14, then from μ₁ = 9 and μ₂ = 10. Do the final groups depend on the starting point, for these six characters?

## For the mini-project: grouping the answers to a survey
Subject 2 of the mini-project applies k-means to a survey. The example below is the one from the course: nine **made-up** answers to two questions, reading hours per week and screen hours per day. For your own data, prepare a CSV table (in Excel: "Save as → CSV UTF-8"), one row per anonymized respondent and one column per numerical question.

- In Colab, tick `load_my_file`: a window opens so that you can choose the file.
- In Jupyter, put the file next to the notebook and type its name in `file_name`.

**Before running:** how many rows and columns does the example table have?

In [ ]:
#@title The course survey (made-up data)
import io

example_csv = """reading_hours_week,screen_hours_day
10,1
12,2
9,1
2,6
1,7
3,6
5,3
6,4
5,4"""
survey = pd.read_csv(io.StringIO(example_csv))
print(survey.shape)

### What you should see
`(9, 2)`: nine respondents and two columns, `reading_hours_week` and `screen_hours_day`.

In [ ]:
#@title TO CHANGE: load your own CSV file (optional)
load_my_file = False  #@param {type:"boolean"}
file_name = "my_survey.csv"  #@param {type:"string"}
separator = ","  #@param [",", ";"]

if load_my_file:
    try:
        from google.colab import files        # this module only exists in Colab
        upload = files.upload()               # a window opens: choose your file
        file_name = list(upload)[0]
    except ImportError:
        print("Not in Colab: the file", file_name, "must be next to the notebook.")
    survey = pd.read_csv(file_name, sep=separator, encoding="utf-8")
print(survey.shape)
print(survey.head())

### What you should see
As long as the box is not ticked, `(9, 2)` and the first five answers of the example. With your own file, note the exact names of two numerical columns: you will type them in the next cell.

**Before running the next cell:** where will the elbow be? Which reader profiles do you expect?

In [ ]:
#@title TO CHANGE: choose two columns and the number of groups k
column_x = "reading_hours_week"  #@param {type:"string"}
column_y = "screen_hours_day"  #@param {type:"string"}
k = 3  #@param {type:"slider", min:2, max:5, step:1}

data = survey[[column_x, column_y]]
for number in range(1, 6):                     # the elbow method
    inertia = KMeans(n_clusters=number, n_init=10, random_state=0).fit(data).inertia_
    print(f"k = {number}: inertia = {inertia:.1f}")
km_survey = KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)
print("Groups:", km_survey.labels_)
print("Centers:", km_survey.cluster_centers_.round(1).tolist())

### What you should see
```
k = 1: inertia = 152.4
k = 2: inertia = 36.7
k = 3: inertia = 9.3
k = 4: inertia = 4.5
k = 5: inertia = 2.3
Groups: [1 1 1 0 0 0 2 2 2]
Centers: [[2.0, 6.3], [10.3, 1.3], [5.3, 3.7]]
```
These are the values from the course. The elbow is at k = 3. The group numbers are arbitrary (they may differ with your version of scikit-learn): it is up to you to interpret each center, for example [10.3, 1.3] for heavy readers, [2.0, 6.3] for heavy screen users and [5.3, 3.7] for intermediate profiles.

**Before running the next cell:** sketch the scatter plot by hand and circle the three groups.

In [ ]:
#@title Draw the groups: one color and one shape per group
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
shapes = ["o", "s", "^", "D", "v"]
plt.figure(figsize=(6, 4))
for g in range(k):
    members = data[km_survey.labels_ == g]           # the respondents in group g
    plt.scatter(members[column_x], members[column_y], color=colors[g], marker=shapes[g], s=70, label=f"Group {g}")
centers = km_survey.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], color="black", marker="x", s=100, label="Centers")
plt.xlabel(column_x)
plt.ylabel(column_y)
plt.title(f"k-means on the survey, k = {k}")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### What you should see
Three well-separated clouds of three points each, with a black cross at the center of each one: heavy readers at the bottom right, heavy screen users at the top left, intermediate profiles in the middle.

**Modify:** move `k` to 2, then to 4, and run both cells again. Which groups merge, which ones split? For your mini-project, if your columns have very different scales (hours and marks out of 20, for example), remember to standardize them before clustering: the `02-questionnaire` notebook shows how.

## To hand in
Answer in writing, in English, in a few lines each:
1. Interpret the line ŷ = 0.9 x + 1.3 from exercise 2 a: what do a and b mean for these made-up students? Why is the prediction for 12 hours a week doubtful?
2. Describe what you observed with η = 0.1, η = 1 and η = 1.1 in gradient descent. In what sense can we say that a model "learns" when it goes down the slope of its error?
3. k-means formed the groups {1, 3, 4} and {9, 10, 14}. Who decides that these are "minor roles" and "main roles": the algorithm or the researcher? Suggest a research question in your own field that clustering could shed light on.

## Key points
- Linear regression predicts a number with the least squares line, a = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)² and b = ȳ − a·x̄; it is only valid within the observed range, and a correlation is not a cause.
- Gradient descent adjusts the parameters step by step, w ← w − η · J′(w): this is how a model learns. A step that is too large diverges, a step that is too small is slow.
- k-means forms groups without labels by alternating assignment and recomputing the centers; the result depends on the starting point and on k (elbow method), and it is up to the researcher to interpret it.

**In the course:** [Some machine learning algorithms](https://progremer04.github.io/canva-au-for-ang-/en/#c3-algorithmes) · Chapter 3 tutorial: [exercise 2 a](https://progremer04.github.io/canva-au-for-ang-/en/#td-ch3-ex2a), [exercise 2 b](https://progremer04.github.io/canva-au-for-ang-/en/#td-ch3-ex2b), [exercise 2 d](https://progremer04.github.io/canva-au-for-ang-/en/#td-ch3-ex2d)